# Structured Streaming

This document is dense, but the concepts are actually simple. It is split into two parts:

1. Simple explanation of every concept
2. Practical hands-on in your Databricks Free Edition

---

## PART 1: Simple Explanation

### Core Idea

**Structured Streaming = process infinite data as if it were a table.**

You write the same DataFrame code you know, and Spark handles the streaming part.

---

### 1. Reading from a Source

Three main source types:

| Source | When to use | Example |
|---|---|---|
| Files (cloud storage) | Ingest JSON/CSV/Parquet | Use Auto Loader |
| Delta Lake | Read changes to another table | `spark.readStream.table("x")` |
| Kafka | Message bus for low latency | `format("kafka")` |

#### Auto Loader (file ingestion)

```python
raw_df = (spark.readStream
    .format("cloudFiles")                              # Auto Loader
    .option("cloudFiles.format", "json")               # file type
    .option("cloudFiles.schemaLocation", "<path>")     # schema inference
    .load(file_path)
)
```

> **Note:** Just configuring a read doesn't start anything. You must trigger an action:
> - `.display()` shows results and starts the stream (in a notebook)
> - `.writeStream...start()` writes continuously

---

### 2. Writing to a Sink

A **sink** is the target where streaming data lands.

| Sink | Use case |
|---|---|
| Delta Lake | Most common: ACID, time travel |
| Kafka | Push to message bus |
| Key-value DB | External system |
| Azure Synapse | Data warehouse |
| Custom (`foreachBatch`) | Anything else |

#### The Four Key Options

Every writer needs:

| Option | Purpose |
|---|---|
| Output mode | Append (default) / Update / Complete |
| Checkpoint location | The stream's "brain" |
| Trigger | When to process |
| Format options | File type, credentials, etc. |

#### Incremental Batch Write (Available Now)

```python
(transformed_df.writeStream
    .trigger(availableNow=True)                        # process new, then stop
    .option("checkpointLocation", checkpoint_path)
    .option("path", target_path)
    .start()
)
```

**What `availableNow=True` does:** processes everything new since the last run, then stops. Perfect for scheduled jobs (like your DLT pipeline).

---

### 3. Reading & Writing Delta Lake

Read from a Delta table as a stream:

```python
spark.readStream.table("<table-name>")
```

Write to a Delta table:

```python
(df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/delta/events/_checkpoints/")
    .toTable("events")
)
```

> **Key point:** Delta + Streaming = exactly-once processing, even with multiple concurrent streams.

---

### 4. Kafka (Message Bus)

Read from Kafka:

```python
df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "<server:ip>")
    .option("subscribe", "<topic>")
    .option("startingOffsets", "latest")   # or "earliest"
    .load()
)
```

Write to Kafka:

```python
(df.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "<server:ip>")
    .option("topic", "<topic>")
    .option("checkpointLocation", "<path>")
    .start()
)
```

**When to use Kafka:** lowest-latency streaming for large datasets.

---

### 5. Unity Catalog + Structured Streaming

- Can stream from managed and external tables in UC
- Can write to UC tables (by table name) or external tables (by path)
- Use the table name for managed tables

```python
spark.readStream.table("my_catalog.my_schema.my_table")   #  UC table
```

---

### 6. Limit Input Rate

Control how much data gets processed per micro-batch:

| Option | Meaning |
|---|---|
| `maxFilesPerTrigger` | Max new files per batch (default 1000) |
| `maxBytesPerTrigger` | Max bytes per batch |

> **If both are set:** the batch stops at whichever limit hits first.

---

### 7. Handling Changes to Source

Structured Streaming assumes the source is **append-only**. If someone modifies or deletes data in the source table, the stream throws an exception.

#### Two Strategies to Handle Changes

**Strategy 1: Restart from Scratch**

- Delete output + checkpoint
- Start over

**Strategy 2: Ignore Certain Changes**

| Option | What it ignores |
|---|---|
| `ignoreDeletes=true` | Deletes at partition boundaries |
| `skipChangeCommits=true` | All deletes AND modifies (superset of `ignoreDeletes`) |

```python
(spark.readStream.format("delta")
    .option("skipChangeCommits", "true")
    .load("path")
)
```

---

### 8. Starting From a Specific Point

Instead of reprocessing the entire table, start from a specific point:

| Option | Meaning |
|---|---|
| `startingVersion=5` | Start from Delta table version 5 |
| `startingTimestamp="2018-10-18"` | Start from a specific date |

Get version numbers with: `DESCRIBE HISTORY your_table`

---

### 9. Stream-Static Joins

Join streaming data with a static Delta table:

```python
streamingDF = spark.readStream.table("orders")   # streaming
staticDF    = spark.read.table("customers")      # static

streamingDF.join(staticDF, "customer_id", "inner")
```

**How it works:** every micro-batch, the static table is re-read at its latest version and joined statelessly.

---

### 10. foreachBatch: The Power Tool

`foreachBatch` lets you run custom logic on each micro-batch's DataFrame.

#### Common Use Cases

| Use case | Why foreachBatch |
|---|---|
| Upsert / MERGE | Standard writes only append |
| Write to multiple sinks | One stream, many targets |
| Write to JDBC (Synapse) | Needs a custom writer |
| Any custom logic | Full DataFrame API |

#### Example: Upsert via MERGE

```python
def upsertToDelta(microBatchDF, batchId):
    (deltaTable.alias("t")
        .merge(microBatchDF.alias("s"), "s.key = t.key")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

(streamingDF.writeStream
    .foreachBatch(upsertToDelta)
    .outputMode("update")
    .start()
)
```

#### Example: Write to Multiple Sinks (with Cache)

```python
def writeToMultiple(df, batchId):
    df.persist()                       # cache to avoid recompute
    df.write.format(...).save(...)     # sink 1
    df.write.format(...).save(...)     # sink 2
    df.unpersist()                     # free memory

streamingDF.writeStream.foreachBatch(writeToMultiple).start()
```

> **Why cache?** Without it, each write recomputes the whole batch. Cache first, then write.

---

### 11. foreach (Row-by-Row)

If `foreachBatch` isn't available, use `foreach` for row-by-row processing:

```python
def processRow(row):
    # Write one row to an external system
    ...

streamingDF.writeStream.foreach(processRow).start()
```

> Slower than `foreachBatch`. Use only when needed.
````

I wrapped the multi-line chained calls in parentheses so they run as valid Python when pasted. Your notes listed Part 2 (hands-on) but didn't include it; send it over when ready.